In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [2]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE); model.eval()
ID2LABEL = {i: l.lower() for i, l in model.config.id2label.items()}
print(DEVICE, ID2LABEL)

def prediction(text_df, batch_size=8):
    df = text_df.copy()
    # Proteccion contra NaN y celdas vacias: el tokenizer se atraganta con floats
    df['text'] = df['text'].fillna("").astype(str).str.strip()
    df = df[df['text'] != ""].reset_index(drop=True)

    probs = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch = df['text'].iloc[i:i+batch_size].tolist()
            inputs = tokenizer(batch, max_length=512, padding=True,
                               truncation=True, return_tensors='pt').to(DEVICE)
            logits = model(**inputs).logits
            probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
            if i % 400 == 0:
                print(f"  {i}/{len(df)}")

    probs = np.vstack(probs)
    for idx, label in ID2LABEL.items():
        df[label.capitalize()] = probs[:, idx]

    df['Net'] = df['Positive'] - df['Negative']
    df['n_words'] = df['text'].str.split().str.len()
    print(f"  truncadas: {(df['n_words'] > 380).sum()}/{len(df)}")
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    return df.drop(columns=['text'])

cpu {0: 'positive', 1: 'negative', 2: 'neutral'}


In [4]:
input_file  = 'C:/Users/PC/Desktop/TFM/BIG DATA/TEXT SENTIMENT/data/ECB_TEXT (2013-2024)_v2.xlsx'
output_file = 'C:/Users/PC/Desktop/TFM/BIG DATA/TEXT SENTIMENT/data/output files/output_finbert_ECB_v2.xlsx'

In [5]:
import re
MAX_W = 300
_SENT = re.compile(r'(?<=[.!?…])\s+(?=[“"A-Z])')

def chunk_text(t, max_w=MAX_W):
    if len(t.split()) <= max_w:
        return [t]
    trozos, actual, n = [], [], 0
    for s in [x for x in _SENT.split(t.strip()) if x]:
        ns = len(s.split())
        if n + ns > max_w and actual:
            trozos.append(" ".join(actual)); actual, n = [], 0
        actual.append(s); n += ns
    if actual:
        trozos.append(" ".join(actual))
    return trozos

def expandir(df):
    filas = []
    for _, r in df.iterrows():
        for j, tr in enumerate(chunk_text(str(r.text)), 1):
            filas.append({**r.to_dict(), "text": tr, "chunk": j})
    out = pd.DataFrame(filas)
    print(f"  {len(df)} -> {len(out)} trozos | aun >380 palabras: {(out.text.str.split().str.len() > 380).sum()}")
    return out

In [6]:
rem = expandir(pd.read_excel(input_file, sheet_name="remarks_para", engine="openpyxl"))
ans = expandir(pd.read_excel(input_file, sheet_name="answers_para", engine="openpyxl"))

  1702 -> 1703 trozos | aun >380 palabras: 0
  3223 -> 3331 trozos | aun >380 palabras: 0


In [7]:
# remarks - por parrafo, troceado   (PRIMERA: mode='w')
rem_out = prediction(rem)
with pd.ExcelWriter(output_file, mode='w', engine="openpyxl") as w:
    rem_out.to_excel(w, sheet_name='remarks_finbert', index=False)

  0/1703
  400/1703
  800/1703
  1200/1703
  1600/1703
  truncadas: 0/1703


In [10]:
# answers - por parrafo, troceado   (SEGUNDA: mode='a')
ans_out = prediction(ans)
with pd.ExcelWriter(output_file, mode='a', engine="openpyxl") as w:
    ans_out.to_excel(w, sheet_name='answers_finbert', index=False)
    if_sheet_exists="replace"

  0/3331
  400/3331
  800/3331
  1200/3331
  1600/3331
  2000/3331
  2400/3331
  2800/3331
  3200/3331
  truncadas: 0/3331


ValueError: Sheet 'answers_finbert' already exists and if_sheet_exists is set to 'error'.

In [11]:
with pd.ExcelWriter(output_file, mode='a', engine="openpyxl",
                    if_sheet_exists="replace") as w:
    ans_out.to_excel(w, sheet_name='answers_finbert', index=False)

In [12]:
print(ans_out.shape)
print(ans_out.columns.tolist())
print(pd.ExcelFile(output_file, engine="openpyxl").sheet_names)

(3331, 10)
['date', 'item', 'order', 'speaker', 'chunk', 'Positive', 'Negative', 'Neutral', 'Net', 'n_words']
['remarks_finbert', 'answers_finbert']


In [13]:
import numpy as np
a = pd.read_excel(input_file, sheet_name="answers", engine="openpyxl")
a["nw"]  = a.text.str.split().str.len()
a["era"] = np.where(a.date.astype(str) < "2019-11-01", "draghi", "lagarde")
print(a.groupby("era").apply(lambda x: (x.nw > 380).mean().round(3), include_groups=False))

era
draghi     0.138
lagarde    0.337
dtype: float64


In [14]:
rem_out = pd.read_excel(output_file, sheet_name="remarks_finbert", engine="openpyxl")

todo = pd.concat([rem_out.assign(seccion="remarks"),
                  ans_out.assign(seccion="answers")], ignore_index=True)

conf = (todo.groupby("date")
            .apply(lambda x: pd.Series({
                "Net_wmean": np.average(x.Net, weights=x.n_words),
                "Net_mean":  x.Net.mean(),
                "n_paras":   len(x),
                "n_words":   x.n_words.sum()}),
                 include_groups=False)
            .reset_index())

conf["era"] = np.where(conf.date < "2019-11-01", "draghi", "lagarde")
print(conf.shape)
print(conf.groupby("era")[["Net_wmean", "Net_mean"]].mean().round(3))

(104, 6)
         Net_wmean  Net_mean
era                         
draghi       0.124     0.121
lagarde      0.047     0.078


In [15]:
conf.to_excel(output_file.replace(".xlsx", "_conference.xlsx"), index=False)

In [16]:
conf["year"] = pd.to_datetime(conf.date).dt.year
print(conf.groupby("year").Net_wmean.mean().round(3))

year
2013    0.032
2014    0.068
2015    0.160
2016    0.212
2017    0.204
2018    0.173
2019    0.107
2020    0.154
2021    0.155
2022    0.053
2023   -0.062
2024   -0.087
Name: Net_wmean, dtype: float64
